## Using arXiv MCP Servers with Python

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
import pymupdf.layout  # activate PyMuPDF-Layout in pymupdf
import pymupdf4llm
from IPython.display import Markdown, display
from datetime import datetime
from rich.console import Console
from rich.panel import Panel
import arxiv
from pydantic import BaseModel, Field

load_dotenv(override=True)

console = Console()

In [ ]:
# ── arXiv Search ───────────────────────────────────────────────────

class ArxivDocument(BaseModel):
    """A single arXiv paper with full metadata."""

    title: str
    entry_id: str
    authors: list[str]
    summary: str
    published: datetime
    updated: datetime
    pdf_url: str
    arxiv_url: str
    primary_category: str
    categories: list[str]
    doi: str | None = None
    comment: str | None = None
    journal_ref: str | None = None


class ArxivDocuments(BaseModel):
    """Collection of arXiv papers from a search."""

    documents: list[ArxivDocument] = Field(default_factory=list)



def search_arxiv(queries: list[str], max_results: int = 5) -> ArxivDocuments:
    """Search arXiv for papers across multiple queries with full metadata."""
    console.print(Panel(
        "\n".join(f"  [cyan]•[/] {q}" for q in queries),
        title="[bold yellow]search_arxiv[/]",
        subtitle=f"max_results={max_results}",
        border_style="yellow",
    ))
    seen_ids: set[str] = set()
    documents: list[ArxivDocument] = []

    arxiv_client = arxiv.Client()   # arxiv.Client(page_size=10, delay_seconds=5.0)

    for query in queries:
        print(f"Searching for {query} Max results {max_results}")
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance,
        )
        for result in arxiv_client.results(search):
            if result.entry_id not in seen_ids:
                seen_ids.add(result.entry_id)
                documents.append(
                    ArxivDocument(
                        title=result.title,
                        entry_id=result.entry_id,
                        authors=[a.name for a in result.authors],
                        summary=result.summary,
                        published=result.published,
                        updated=result.updated,
                        pdf_url=result.pdf_url,
                        arxiv_url=result.entry_id,
                        primary_category=result.primary_category,
                        categories=result.categories,
                        doi=result.doi,
                        comment=result.comment,
                        journal_ref=result.journal_ref,
                    )
                )

    console.print(f"  [green]✓[/] Found [bold]{len(documents)}[/] arXiv papers\n")
    return ArxivDocuments(documents=documents)

In [ ]:
aarxiv_docs = search_arxiv(["machine learning", "AI engineering"])
arxiv_docs = aarxiv_docs.documents
print(f"arXiv docs: {len(arxiv_docs)}")
print(f"arXiv doc: {arxiv_docs[0]}")
print(f"arXiv doc entry_id: {arxiv_docs[0].entry_id}")
print(f"arxiv doc doi: {arxiv_docs[0].doi}")
print(f"arXiv doc categories: {arxiv_docs[0].categories}")
print(f"arXiv doc primary category: {arxiv_docs[0].primary_category}")
print(f"arXiv doc journal ref: {arxiv_docs[0].journal_ref}")

In [ ]:
arxiv_mcp_params = {
    "command": "uv",
    "args": [
        "tool",
        "run",
        "arxiv-mcp-server",
        "--storage-path", os.getenv("ARXIV_STORAGE_PATH")
    ]
}

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=30) as server:
    mcp_prompts = await server.list_prompts()

mcp_prompts

In [ ]:
print("=== Available Tools ===")
for tool in mcp_tools:
    print(f"\n  Tool: {tool.name}")
    print(f"  Description: {tool.description}")
    print(f"  Input Schema: {tool.inputSchema}")

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=120) as server:
    search_result = await server.call_tool("search_papers", {
        "query": "transformer architecture""",
        "max_results": 10,
        # "date_from": "2023-01-01",
        "categories": ["cs.AI", "cs.LG"]
    })
    print(search_result)

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=120) as server:
    download_result = await server.call_tool("download_paper", {
        "paper_id": "2504.12345"
    })
    print(download_result)

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=30) as server:
    prompt_result = await server.get_prompt("deep-paper-analysis", arguments={"paper_id": "2504.12345"})

print(f"Description: {prompt_result.description}")
for msg in prompt_result.messages:
    print(f"\nRole: {msg.role}")
    print(f"Content: {msg.content.text}")

In [ ]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as file_server:
    file_tools = await file_server.list_tools()

file_tools

In [ ]:
model = "gpt-4.1-mini"
instructions = """You are a research assistant that finds and analyzes academic papers. 
Use the arxiv tools to search, download, and read papers. 
Provide thorough analysis of methodology and findings.
When you need to write files, you do that inside the sandbox folder only.

IMPORTANT: The arXiv API has strict rate limits. When searching:
- Make ONE well-crafted search query rather than multiple attempts.
- If you get a rate limit error, wait and try again once — do NOT retry rapidly.
- Use specific, targeted queries with category filters to get good results in a single call.
- Limit results to max_results=5 to reduce API load."""

In [ ]:
async with MCPServerStdio(params=arxiv_mcp_params, client_session_timeout_seconds=120) as arxiv_server:
    async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as file_server:
        agent = Agent(name="research_assistant_agent", instructions=instructions, model=model, mcp_servers=[arxiv_server, file_server])
        with trace("Research Assistant"):
            result = await Runner.run(agent, "Find recent papers on speech synthesis for text to speech "
                    "and summarize the top 3 results. Write the summary to a file called analysis_summary.md")
        display(Markdown(result.final_output))